# U26 — Model Training Lab

### Real-world brief: predicting turbine-blade fatigue cracks

You have fatigue-test data for **1,800 turbine blades** and need a model that flags blades likely to crack. But this lab isn't really about the model — it's about **how to train one well**. You'll build a training loop **from scratch** (so you see forward → loss → backward → step), then run the experiments every practitioner should know: finding the **learning rate**, the effect of **batch size**, diagnosing and fixing **overfitting** with regularization and **early stopping**, a small **hyperparameter search**, and **debugging** tricks.

**Resource provided:** `turbine_blades.csv` (1,800 blades, 10 features + crack label). Built with numpy (the training loop) + scikit-learn (splitting & scaling) + matplotlib.

_Phase G — Practice._

#objectives

Implement a gradient-descent training loop from scratch

Split data without leakage; fit preprocessing on train only

Find a good learning rate and understand batch size

Diagnose overfitting and fix it with regularization + early stopping

Run a small hyperparameter search and debug training failures

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: build the dataset if missing ===
import os
import numpy as np
import pandas as pd


def build_blades(path="turbine_blades.csv", seed=261, verbose=False):
    """Turbine-blade fatigue test data for the model-training lab (U26). Each row is a blade
    that was run on a test rig; the target is whether a fatigue crack was detected. The signal
    is real but noisy, and there are enough features to overfit a flexible model — so learning
    rate, regularization and early stopping all matter.

      features: alloy_grade, coating_um, operating_temp_c, vibration_g, rpm, hours_in_service,
                prior_repairs, inspection_score, blade_length_mm, mounting_torque_nm
      target:   crack_detected (0/1)
    """
    rng = np.random.default_rng(seed)
    n = 1800
    alloy = rng.choice([1, 2, 3], n, p=[0.4, 0.4, 0.2])              # 3 = best alloy
    coating = np.clip(rng.normal(120, 30, n), 40, None)
    op_temp = rng.normal(640, 45, n)
    vibration = np.clip(rng.normal(3.0, 1.1, n), 0.2, None)
    rpm = rng.normal(9500, 900, n)
    hours = rng.uniform(0, 24000, n)
    prior_repairs = rng.poisson(0.8, n)
    inspection = np.clip(rng.normal(82, 9, n), 40, 100)
    length = rng.normal(180, 12, n)
    torque = rng.normal(95, 10, n)

    # genuine stress drivers -> crack risk (sharp, learnable)
    drive = (0.00007 * np.maximum(hours - 12000, 0)
             + 0.5 * np.maximum(vibration - 3.5, 0)
             + 0.010 * np.maximum(op_temp - 660, 0)
             + 0.45 * prior_repairs
             - 0.6 * (alloy - 1)
             - 0.02 * (inspection - 82))
    thr = np.quantile(drive, 0.72)
    p = 1 / (1 + np.exp(-2.6 * (drive - thr)))
    crack = (rng.random(n) < p).astype(int)

    df = pd.DataFrame({
        "alloy_grade": alloy, "coating_um": coating.round(1), "operating_temp_c": op_temp.round(1),
        "vibration_g": vibration.round(2), "rpm": rpm.round(0), "hours_in_service": hours.round(0),
        "prior_repairs": prior_repairs, "inspection_score": inspection.round(1),
        "blade_length_mm": length.round(1), "mounting_torque_nm": torque.round(1),
        "crack_detected": crack,
    })
    df.to_csv(path, index=False)
    if verbose:
        print("shape:", df.shape, "| crack rate:", round(df.crack_detected.mean(), 3))
        print(df.head(3).to_string(index=False))
    return df

if not os.path.exists('turbine_blades.csv'):
    build_blades(); print('Generated turbine_blades.csv')
else:
    print('Found turbine_blades.csv')

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
df = pd.read_csv('turbine_blades.csv')
print('shape:', df.shape, '| crack rate:', round(df.crack_detected.mean(), 3))
df.head(3)

In [ ]:
# -----------------------------------------------------------
# 🔹 0. SPLIT into train / val / test, then SCALE (fit on train only)
# Fitting the scaler on train only = no leakage from val/test into training.
# -----------------------------------------------------------
X = df.drop(columns='crack_detected').values.astype(float); y = df.crack_detected.values
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
Xval, Xte, yval, yte = train_test_split(Xtmp, ytmp, test_size=0.5, random_state=0, stratify=ytmp)
scaler = StandardScaler().fit(Xtr)              # <-- fit on TRAIN only
Xtr_s, Xval_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xval), scaler.transform(Xte)
print('train/val/test sizes:', len(Xtr), len(Xval), len(Xte))

#1. The training loop, from scratch

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. Logistic regression by gradient descent
# forward (predict) -> loss (BCE) -> backward (gradient) -> step (update)
# -----------------------------------------------------------
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def bce_loss(w, X, y, lam=0.0):
    p = sigmoid(X @ w)
    data = -np.mean(y * np.log(p + 1e-9) + (1 - y) * np.log(1 - p + 1e-9))
    return data + lam * np.sum(w[1:] ** 2)            # L2 penalty (not on bias)

def train(X, y, lr=0.3, epochs=400, lam=0.0, Xval=None, yval=None):
    Xb = np.c_[np.ones(len(X)), X]; w = np.zeros(Xb.shape[1])
    tr_hist, val_hist = [], []
    for ep in range(epochs):
        p = sigmoid(Xb @ w)                            # forward
        grad = Xb.T @ (p - y) / len(y)                 # backward (gradient of BCE)
        grad[1:] += 2 * lam * w[1:]                    # + regularization gradient
        w -= lr * grad                                 # step
        tr_hist.append(bce_loss(w, Xb, y, lam))
        if Xval is not None:
            val_hist.append(bce_loss(w, np.c_[np.ones(len(Xval)), Xval], yval, lam))
    return w, tr_hist, val_hist

def accuracy(w, X, y):
    return float(((sigmoid(np.c_[np.ones(len(X)), X] @ w) > 0.5).astype(int) == y).mean())

w, tr_hist, _ = train(Xtr_s, ytr, lr=0.3, epochs=400)
print('final train loss:', round(tr_hist[-1], 4))
print('train acc:', round(accuracy(w, Xtr_s, ytr), 3), '| val acc:', round(accuracy(w, Xval_s, yval), 3))

In [ ]:
plt.figure(figsize=(7, 3.4)); plt.plot(tr_hist, color='#2D6A9F')
plt.xlabel('epoch'); plt.ylabel('training loss (BCE)'); plt.title('The loss goes down — learning!')
plt.tight_layout(); plt.show()

#### 🧪 EXERCISE 1 — Read the loop
1. The four lines inside the loop are forward, backward, regularization, step. In a comment, label each line with which stage it is.
2. Add an argument `verbose` that prints the loss every 100 epochs, and run it — confirm the loss decreases monotonically here (a convex problem).

In [ ]:
# 1. label the four stages (comment)
# 2. add a verbose print every 100 epochs and run
# YOUR CODE HERE

#2. The learning rate — the most important knob

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. Same model, same data — only the learning rate changes
# -----------------------------------------------------------
plt.figure(figsize=(7.5, 4))
for lr in [0.001, 0.01, 0.1, 1.0, 30.0]:
    _, h, _ = train(Xtr_s, ytr, lr=lr, epochs=150)
    plt.plot(h, label=f'lr={lr}')
plt.xlabel('epoch'); plt.ylabel('training loss'); plt.ylim(0.4, 1.0)
plt.title('Learning rate controls how fast (and whether) you converge')
plt.legend(); plt.tight_layout(); plt.show()
print('Too low (0.001): crawls. Good (0.1-1.0): fast, smooth. Too high (30): overshoots / unstable.')

#### 🧪 EXERCISE 2 — Find the best rate
1. Sweep at least six learning rates and, for each, record the **validation loss** after a fixed epoch budget. Print them and pick the best.
2. In a comment, explain why the learning rate is usually the first hyperparameter you tune.

In [ ]:
# 1. LR sweep on validation loss; pick the best
# YOUR CODE HERE

# 2. why tune LR first: ...   (comment)

#3. Batch size & epochs

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. FULL-BATCH vs MINI-BATCH (stochastic) gradient descent
# -----------------------------------------------------------
def train_sgd(X, y, lr=0.3, epochs=60, batch=32, seed=0):
    rng = np.random.default_rng(seed); Xb = np.c_[np.ones(len(X)), X]; w = np.zeros(Xb.shape[1]); hist = []
    for ep in range(epochs):
        idx = rng.permutation(len(X))
        for i in range(0, len(X), batch):
            j = idx[i:i + batch]; p = sigmoid(Xb[j] @ w)
            w -= lr * (Xb[j].T @ (p - y[j]) / len(j))
        hist.append(bce_loss(w, Xb, y))
    return w, hist

_, full, _ = train(Xtr_s, ytr, lr=0.3, epochs=60)
_, mb32 = train_sgd(Xtr_s, ytr, lr=0.3, epochs=60, batch=32)
_, mb8 = train_sgd(Xtr_s, ytr, lr=0.3, epochs=60, batch=8)
plt.figure(figsize=(7.5, 3.6))
plt.plot(full, label='full-batch'); plt.plot(mb32, label='mini-batch 32'); plt.plot(mb8, label='mini-batch 8')
plt.xlabel('epoch'); plt.ylabel('training loss'); plt.title('Smaller batches: noisier but more updates/epoch')
plt.legend(); plt.tight_layout(); plt.show()
print('Mini-batches take many small steps per epoch -> faster early progress, with more noise.')

#### 🧪 EXERCISE 3 — Batch size trade-off
1. Time how long `batch=8` vs `batch=256` take for the same number of epochs, and compare their validation accuracy.
2. In a comment, summarise the trade-off (small = noisier gradients, can generalise better, slower per epoch in Python; large = smoother, faster, more memory).

In [ ]:
# 1. compare batch=8 vs batch=256 (time + val accuracy)
# YOUR CODE HERE

# 2. batch-size trade-off: ...   (comment)

#4. Overfitting & regularization

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. Force overfitting: many features, few samples
# Expand to 65 polynomial features and train on just 90 blades.
# -----------------------------------------------------------
poly = PolynomialFeatures(2, include_bias=False)
sub = np.random.RandomState(0).choice(len(Xtr), 90, replace=False)
Xtr_sm, ytr_sm = Xtr[sub], ytr[sub]
scp = StandardScaler().fit(poly.fit_transform(Xtr_sm))
Xtr_p = scp.transform(poly.transform(Xtr_sm)); Xval_p = scp.transform(poly.transform(Xval))
print('expanded to', Xtr_p.shape[1], 'features on', len(Xtr_sm), 'training samples')

w0, tr0, va0 = train(Xtr_p, ytr_sm, lr=0.3, epochs=2000, lam=0.0, Xval=Xval_p, yval=yval)
print(f'NO regularization : train acc {accuracy(w0, Xtr_p, ytr_sm):.3f} | val acc {accuracy(w0, Xval_p, yval):.3f}  <- big gap = overfit')
wL, trL, vaL = train(Xtr_p, ytr_sm, lr=0.3, epochs=2000, lam=0.05, Xval=Xval_p, yval=yval)
print(f'WITH L2 (lam=0.05): train acc {accuracy(wL, Xtr_p, ytr_sm):.3f} | val acc {accuracy(wL, Xval_p, yval):.3f}  <- gap closes, val improves')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
ax[0].plot(tr0, label='train'); ax[0].plot(va0, label='val'); ax[0].set_title('No regularization (overfits)')
ax[1].plot(trL, label='train'); ax[1].plot(vaL, label='val'); ax[1].set_title('With L2 (val tracks train)')
for a in ax: a.set_xlabel('epoch'); a.legend()
ax[0].set_ylabel('loss'); plt.tight_layout(); plt.show()
print('Left: val loss rises while train keeps falling (memorising). Right: the gap stays small.')

#### 🧪 EXERCISE 4 — Tune the regularization strength
1. Sweep `lam` over e.g. [0, 0.001, 0.01, 0.05, 0.2, 1.0] on the expanded features and plot validation accuracy vs `lam`. Find the sweet spot.
2. In a comment, explain the bias-variance intuition: too little regularization overfits, too much underfits.

In [ ]:
# 1. sweep lam; plot val accuracy vs lam
# YOUR CODE HERE

# 2. bias-variance intuition: ...   (comment)

#5. Early stopping

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. Stop when validation loss stops improving; keep the BEST weights
# -----------------------------------------------------------
def train_early_stop(X, y, Xval, yval, lr=0.3, max_epochs=3000, patience=50):
    Xb = np.c_[np.ones(len(X)), X]; w = np.zeros(Xb.shape[1])
    best_loss, best_w, best_ep, wait = np.inf, w.copy(), 0, 0
    val_hist = []
    for ep in range(max_epochs):
        p = sigmoid(Xb @ w); w -= lr * (Xb.T @ (p - y) / len(y))
        vl = bce_loss(w, np.c_[np.ones(len(Xval)), Xval], yval); val_hist.append(vl)
        if vl < best_loss - 1e-5:
            best_loss, best_w, best_ep, wait = vl, w.copy(), ep, 0
        else:
            wait += 1
            if wait >= patience: break        # no improvement for `patience` epochs
    return best_w, best_ep, val_hist

bw, best_ep, vh = train_early_stop(Xtr_p, ytr_sm, Xval_p, yval)
print(f'stopped at epoch {len(vh)}, best validation epoch was {best_ep}')
print('val acc with best weights:', round(accuracy(bw, Xval_p, yval), 3))
plt.figure(figsize=(7, 3.4)); plt.plot(vh, color='#C0392B'); plt.axvline(best_ep, ls='--', color='#1f6b33', label='best epoch')
plt.xlabel('epoch'); plt.ylabel('validation loss'); plt.title('Early stopping keeps the best, not the last'); plt.legend()
plt.tight_layout(); plt.show()

#### 🧪 EXERCISE 5 — Patience matters
1. Try `patience=5` and `patience=200`. How does the stopping epoch and final val accuracy change?
2. In a comment, explain why we restore the **best** checkpoint rather than using the weights from the final epoch.

In [ ]:
# 1. compare patience = 5 vs 200
# YOUR CODE HERE

# 2. why restore best, not last: ...   (comment)

#6. Tuning, debugging & reproducibility

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. A small RANDOM SEARCH over (lr, lambda), scored on validation
# Tune on validation; touch test only ONCE at the very end.
# -----------------------------------------------------------
rng = np.random.default_rng(0); trials = []
for _ in range(12):
    lr = float(10 ** rng.uniform(-2, 0)); lam = float(10 ** rng.uniform(-3, -0.3))
    w_, _, _ = train(Xtr_p, ytr_sm, lr=lr, epochs=1500, lam=lam)
    trials.append((round(lr, 4), round(lam, 4), round(accuracy(w_, Xval_p, yval), 3)))
res = pd.DataFrame(trials, columns=['lr', 'lambda', 'val_acc']).sort_values('val_acc', ascending=False)
print(res.head(6).to_string(index=False))
best = res.iloc[0]
print(f'\nbest config: lr={best["lr"]}, lambda={best["lambda"]}, val_acc={best["val_acc"]}')

In [ ]:
# -----------------------------------------------------------
# 🔹 6B. DEBUGGING trick: a healthy model can overfit a TINY batch
# If it can't memorise 12 samples, something is broken (data/loss/gradient).
# -----------------------------------------------------------
tiny = np.random.RandomState(1).choice(len(Xtr_s), 12, replace=False)
w_tiny, _, _ = train(Xtr_s[tiny], ytr[tiny], lr=0.5, epochs=3000)
print('accuracy on the 12-sample batch:', round(accuracy(w_tiny, Xtr_s[tiny], ytr[tiny]), 3),
      '(should be ~1.0 -> gradients & loss are wired correctly)')

# Reproducibility: same seed -> identical result
a, _ = train_sgd(Xtr_s, ytr, seed=42); b, _ = train_sgd(Xtr_s, ytr, seed=42)
print('two runs with the same seed identical?', bool(np.allclose(a, b)))

#### 🧪 EXERCISE 6 — Final honest evaluation
1. Take your **best config** from the random search, retrain on the expanded features, and evaluate on the **test set** — which you have not touched until now. Report test accuracy.
2. In a comment, explain why touching the test set only once gives an honest estimate of real-world performance.

In [ ]:
# 1. retrain best config; evaluate on the held-out TEST set (once)
# YOUR CODE HERE

# 2. why test is touched only once: ...   (comment)

#📘 Summary

| Topic | What you did |
| ----- | ------------ |
| Training loop | built forward → loss → backward → step by hand |
| Learning rate | saw too-low/too-high/just-right loss curves |
| Batch size | compared full-batch vs mini-batch SGD |
| Overfitting | forced it, then closed the gap with L2 |
| Early stopping | kept the best checkpoint, not the last |
| Tuning & debugging | random search, tiny-batch sanity check, seeds |

**Core lesson:** training is a craft. Get the **learning rate** right, watch **validation** (not train) loss, **regularize** and **early-stop** to fight overfitting, **search** hyperparameters honestly on validation, and keep everything **reproducible** — then judge once on the test set.

**You've now completed the hands-on labs for the course** — from data engineering and MLOps to reinforcement learning and the craft of training itself.